# **Construindo o catalogo  Usuários X Filmes**

In [ ]:
import pandas as pd
import numpy as np


# Definindo a matriz de ratings (NaN = não avaliado)
ratings_data = {
    'Star Wars':        [5, 2, 3, 4, 1, 5, 5, 3, 2, 4],
    'Matrix':           [5, 2, 4, 3, 1, 4, 5, 3, 2, 5],
    'Interestelar':     [4, 3, 3, 4, 2, 5, 5, 4, 3, 4],
    'Titanic':          [2, 5, 3, 2, 1, 3, 2, 3, 5, 3],
    'Diário de uma Paixão': [1, 5, 2, 2, 1, 2, 1, 2, 5, 2],
    'O Poderoso Chefão':[4, 3, 5, 3, 4, 5, 4, 5, 3, 4],
    'Pulp Fiction':     [3, 2, 5, 3, 4, 4, 3, 5, 2, 5],
    'Toy Story':        [3, 3, 2, 5, 2, 4, 3, 2, 3, 4],
    'Procurando Nemo':  [2, 3, 2, 5, 1, 4, 2, 2, 3, 3],
    'O Iluminado':      [3, 2, 4, 2, 5, 4, 3, 4, 2, 4]
}

usuarios = ['Ana', 'Bruno', 'Carla', 'Diego', 'Elisa',
            'Felipe', 'Gabriela', 'Henrique', 'Isabela', 'João']

df = pd.DataFrame(ratings_data, index=usuarios)

## **introduzindo valores faltantes**

In [ ]:
total_celulas = df.shape[0] * df.shape[1]
ausentes = df.isna().sum().sum()

In [ ]:
np.random.seed(42)
df_com_nan = df.copy()
for _ in range(20):  # removendo ~20% dos valores
    u = np.random.randint(0, 10)
    i = np.random.randint(0, 10)
    df_com_nan.iloc[u, i] = np.nan

## **Exibindo o dataframe final**

In [ ]:
print("\nDataFrame com valores ausentes (simulando esparsidade):")
print(df_com_nan)


DataFrame com valores ausentes (simulando esparsidade):
          Star Wars  Matrix  Interestelar  Titanic  Diário de uma Paixão  \
Ana             5.0     5.0           4.0      2.0                   1.0   
Bruno           2.0     2.0           3.0      NaN                   5.0   
Carla           3.0     4.0           3.0      3.0                   2.0   
Diego           4.0     3.0           4.0      2.0                   2.0   
Elisa           NaN     1.0           NaN      1.0                   1.0   
Felipe          5.0     NaN           5.0      3.0                   NaN   
Gabriela        5.0     5.0           5.0      NaN                   NaN   
Henrique        3.0     3.0           NaN      3.0                   NaN   
Isabela         NaN     2.0           NaN      5.0                   5.0   
João            4.0     5.0           NaN      3.0                   2.0   

          O Poderoso Chefão  Pulp Fiction  Toy Story  Procurando Nemo  \
Ana                     4.0      

#

## **Desenvolvendo o algoritmo SVD**

In [ ]:
import numpy as np
import pandas as pd
from numpy.linalg import svd


### **Matriz de ratings**

In [ ]:
R = df_com_nan.values.copy()


In [ ]:
print(R)

[[ 5.  5.  4.  2.  1.  4.  3.  3.  2.  3.]
 [ 2.  2.  3. nan  5.  3.  2. nan  3.  2.]
 [ 3.  4.  3.  3.  2.  5. nan  2.  2.  4.]
 [ 4.  3.  4.  2.  2.  3.  3. nan  5.  2.]
 [nan  1. nan  1.  1.  4.  4.  2.  1.  5.]
 [ 5. nan  5.  3. nan  5.  4.  4.  4.  4.]
 [ 5.  5.  5. nan nan  4.  3.  3.  2. nan]
 [ 3.  3. nan  3. nan  5.  5.  2.  2.  4.]
 [nan  2. nan  5.  5.  3. nan  3.  3.  2.]
 [ 4.  5. nan  3.  2. nan  5.  4.  3.  4.]]


In [ ]:
# Máscara de valores observados
mask_observed = ~np.isnan(R)

In [ ]:
print(mask_observed)

[[ True  True  True  True  True  True  True  True  True  True]
 [ True  True  True False  True  True  True False  True  True]
 [ True  True  True  True  True  True False  True  True  True]
 [ True  True  True  True  True  True  True False  True  True]
 [False  True False  True  True  True  True  True  True  True]
 [ True False  True  True False  True  True  True  True  True]
 [ True  True  True False False  True  True  True  True False]
 [ True  True False  True False  True  True  True  True  True]
 [False  True False  True  True  True False  True  True  True]
 [ True  True False  True  True False  True  True  True  True]]


### **Calculo da média do item**

In [ ]:
col_means = np.nanmean(R, axis=0)

In [ ]:
col_means

array([3.875     , 3.33333333, 4.        , 2.75      , 2.57142857,
       4.        , 3.625     , 2.875     , 2.7       , 3.33333333])

In [ ]:
R_imputed = np.where(np.isnan(R), col_means, R)

In [ ]:
print("Matriz imputada")
print(R_imputed[:5])


Matriz imputada
[[5.    5.    4.    2.    1.    4.    3.    3.    2.    3.   ]
 [2.    2.    3.    2.75  5.    3.    2.    2.875 3.    2.   ]
 [3.    4.    3.    3.    2.    5.    3.625 2.    2.    4.   ]
 [4.    3.    4.    2.    2.    3.    3.    2.875 5.    2.   ]
 [3.875 1.    4.    1.    1.    4.    4.    2.    1.    5.   ]]


### **Centralizar((subtrair a média global)**

In [ ]:
mu = R_imputed.mean()
R_centered = R_imputed - mu

##  **Aplicar SVD truncada**

In [ ]:
k = 3  # número de fatores latentes
U, sigma, Vt = svd(R_centered, full_matrices=False)


In [ ]:
# Truncar em k dimensões
U_k = U[:, :k]
sigma_k = np.diag(sigma[:k])
Vt_k = Vt[:k, :]

In [ ]:
# Reconstruir a matriz aproximada
R_approx = U_k @ sigma_k @ Vt_k + mu

In [ ]:
print(R_approx[:4])

[[4.9530942  4.46771465 4.44930015 1.84696815 1.01963725 4.09047619
  3.49727003 3.21357742 2.95822963 3.6161559 ]
 [2.28626613 1.54462253 2.60705793 3.15539955 4.32956118 2.35500769
  2.43547044 2.9394829  3.60211697 2.65619257]
 [3.49318414 3.34993027 3.64860542 2.90128557 2.28511421 4.39713631
  4.25105169 2.71686457 1.97658848 4.39020226]
 [4.2365872  3.27241673 3.80515127 1.8918281  2.27908652 2.69097115
  2.20514884 3.36427755 4.09715303 2.39458177]]


In [ ]:
print(f"\nDimensões: U={U_k.shape}, Σ={sigma_k.shape}, Vᵀ={Vt_k.shape}")
print(f"Variância explicada pelos {k} fatores: "
      f"{sigma[:k].sum() / sigma.sum():.2%}")


Dimensões: U=(10, 3), Σ=(3, 3), Vᵀ=(3, 10)
Variância explicada pelos 3 fatores: 57.43%


## **Gerar recomendações**

In [ ]:
def recomendar_svd(df_original, R_approx, usuario, top_n=3):
    """Recomenda filmes não avaliados pelo usuário."""

    #Retorna os filmes disponíveis no catálogo.
    filmes = df_original.columns.tolist()

    #Identifica o indicie do usuário
    idx_user = df_original.index.get_loc(usuario)



    # Ratings originais do usuário
    ratings_originais = df_original.iloc[idx_user].values
    print(  ratings_originais)



    # Previsões para todos os filmes
    previsoes = R_approx[idx_user]
    print(previsoes)

    # Máscara: apenas filmes NÃO avaliados
    nao_avaliados = np.isnan(ratings_originais)

    # Ordenar previsões
    candidatos = [(filmes[i], previsoes[i])
                  for i in range(len(filmes)) if nao_avaliados[i]]


    candidatos.sort(key=lambda x: x[1], reverse=True)

    return candidatos[:top_n]

In [ ]:

print("\n" + "=" * 60)
print("RECOMENDAÇÕES COM SVD CLÁSSICA (imputação)")
print("=" * 60)
for usuario in ['Bruno']:
    print(f"\n Recomendações para {usuario}:")
    for filme, nota in recomendar_svd(df_com_nan, R_approx, usuario):
        print('fim execução')
        print(f"   → {filme}: {nota:.2f}")



RECOMENDAÇÕES COM SVD CLÁSSICA (imputação)

 Recomendações para Bruno:
[ 2.  2.  3. nan  5.  3.  2. nan  3.  2.]
[2.28626613 1.54462253 2.60705793 3.15539955 4.32956118 2.35500769
 2.43547044 2.9394829  3.60211697 2.65619257]
fim execução
   → Titanic: 3.16
fim execução
   → Toy Story: 2.94
